# Chapter 6: advanced matrix multiplication

We will use a **Dell Pro Max with GB10** as the main experiment machine. You can also use another supported NVIDIA GPU, locally or in the cloud. These examples target Linux, CUDA PyTorch, Triton, and compute capability 8.0 or later. CPU, Apple GPU, and AMD execution are outside this notebook's implementation.

All kernels, wrappers, correctness checks, and timing functions are defined in this notebook. Run the cells in order. The runner checks the GPU and APIs, records unavailable optional examples as skips, and runs correctness checks before timing each provider. GB10 requires a software build that supports its ARM CPU and SM121 GPU; use a GB10-compatible environment rather than copying the archived H100 environment. The October 13 GB10 run completed with six resource-limited configurations skipped. The recorded runtime was PyTorch 2.13.0+cu129 and Triton 3.7.1. These observations cover that run and its checked inputs; check a new environment before interpreting its timings. The Section 6.1 H100 archive is separate evidence.

Block scaling always includes **explicit software reconstruction to BF16**. The optional `tl.dot_scaled` provider has target-dependent lowering: H100 uses emulation; the recorded GB10 kernel also uses BF16 emulation, as verified from its PTX. This is a result for that compiler path, not a claim about every GB10 implementation.

## Set up the environment

The imports require CUDA-enabled PyTorch and Triton in a Linux environment.
The device check reports which optional examples can be attempted. After changing
a definition, restart the kernel and run all cells to rebuild the kernels and
their dependencies together.

In [ ]:
# ch06-source: imports
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
import hashlib
import json
import shutil
import statistics
import time

import torch
import triton
import triton.language as tl
import triton.testing
from triton.runtime.errors import OutOfResources

CH06_SOURCE_CELLS = ['imports', 'target-policy', 'settings', 'baseline', 'dense-configuration', 'pointer-kernel', 'descriptor-kernel', 'printed-persistent', 'split-k', 'grouped', 'block-scaled', 'block-sparse', 'measurement', 'recorder', 'correctness', 'dense-comparison', 'split-comparison', 'grouped-comparison', 'scaled-comparison', 'sparse-comparison', 'runner']

In [ ]:
# ch06-source: target-policy
def feature_plan(capability, *, has_descriptors, has_dot_scaled,
                 descriptors="auto", dot_scaled="auto"):
    """Pure policy; compilation and correctness still run on the user's GPU."""
    if capability[0] < 8:
        raise RuntimeError("these FP16/BF16 examples require NVIDIA CC 8.0+")
    plan = {}
    for name, mode, available in (
        ("descriptors", descriptors, has_descriptors),
        ("dot_scaled", dot_scaled, has_dot_scaled),
    ):
        if mode not in ("auto", "on", "off"):
            raise ValueError(f"invalid {name} mode: {mode}")
        # Keep the optional low-precision kernel to SM90+ targets. SM80/SM86
        # still run the explicit reconstruction example, including timings.
        reason = None
        if mode == "off":
            reason = "disabled by the reader"
        elif capability[0] < 9:
            reason = "this optional example requires CC 9.0+"
        elif not available:
            reason = "the installed Triton lacks the required API"
        if mode == "on" and reason:
            raise RuntimeError(f"{name}: {reason}")
        plan[name] = dict(enabled=reason is None, requested=mode,
                          reason=reason, validation="GPU run required")
    return plan


def inspect_target(*, descriptors="auto", dot_scaled="auto"):
    import platform
    import torch
    import triton
    import triton.language as tl

    if not torch.cuda.is_available() or torch.version.hip:
        raise RuntimeError(
            "Use Linux with NVIDIA CUDA PyTorch and Triton. "
            "CPU, Apple GPU, and AMD execution are not implemented here.")
    if not hasattr(torch, "float8_e4m3fn"):
        raise RuntimeError(
            "This PyTorch build lacks float8_e4m3fn, which the explicit "
            "block-scale reconstruction example needs for byte conversion.")
    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)
    capability = torch.cuda.get_device_capability(device)
    plan = feature_plan(
        capability,
        has_descriptors=(hasattr(tl, "make_tensor_descriptor")
                         and hasattr(triton, "set_allocator")),
        has_dot_scaled=(hasattr(tl, "dot_scaled")
                        and hasattr(torch, "float8_e4m3fn")),
        descriptors=descriptors, dot_scaled=dot_scaled)
    return dict(
        chapter_machine="Dell Pro Max with GB10",
        gpu=props.name, device_index=device, sms=props.multi_processor_count,
        capability=list(capability), cpu_arch=platform.machine(),
        os=platform.platform(), python=platform.python_version(),
        torch=torch.__version__, triton=triton.__version__,
        cuda=torch.version.cuda,
        shared_memory_per_block=getattr(props, "shared_memory_per_block", None),
        shared_memory_per_block_optin=getattr(
            props, "shared_memory_per_block_optin", None),
        torch_compiled_arches=torch.cuda.get_arch_list(),
        features=plan,
        block_scaled_path="explicit BF16 reconstruction on every target; "
                          "optional tl.dot_scaled lowering is target dependent",
        instructions_verified=False)

In [ ]:
# ch06-source: settings
# auto: attempt optional examples when the minimum requirements are present.
# on: require an optional example; off: omit it and record that choice.
DESCRIPTORS = "auto"
DOT_SCALED = "auto"
OUTPUT_ROOT = Path.cwd() / "output"

target = inspect_target(descriptors=DESCRIPTORS, dot_scaled=DOT_SCALED)
print(json.dumps(target, indent=2))

## 6.1: Chapter 5 baseline

We keep the Chapter 5 autotuned multiplication as the baseline for the dense comparisons.

In [ ]:
# ch06-source: baseline
def matmul_autotune_configs():
    return [
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 256, "BLOCK_K": 64, "GROUP_M": 8},
            num_warps=8, num_stages=3,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 256, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4, num_stages=4,
        ),
    ]



@triton.autotune(
    configs=matmul_autotune_configs(),
    key=["M", "N", "K"],
)
@triton.jit
def optimized_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    pid_in_group = pid % num_pid_in_group
    pid_m = first_pid_m + pid_in_group % group_size_m
    pid_n = pid_in_group // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_m = tl.max_contiguous(
        tl.multiple_of(offs_m, BLOCK_M), BLOCK_M
    )
    offs_n = tl.max_contiguous(
        tl.multiple_of(offs_n, BLOCK_N), BLOCK_N
    )

    a_ptrs = (
        a_ptr
        + offs_m[:, None] * stride_am
        + offs_k[None, :] * stride_ak
    )
    b_ptrs = (
        b_ptr
        + offs_k[:, None] * stride_bk
        + offs_n[None, :] * stride_bn
    )

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k_tile in range(0, tl.cdiv(K, BLOCK_K)):
        valid_k = offs_k < K - k_tile * BLOCK_K
        a_mask = (offs_m[:, None] < M) & valid_k[None, :]
        b_mask = valid_k[:, None] & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        acc = tl.dot(a, b, acc)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = (
        c_ptr
        + offs_m[:, None] * stride_cm
        + offs_n[None, :] * stride_cn
    )
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)



def optimized_matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    if a.ndim != 2 or b.ndim != 2:
        raise ValueError("a and b must both be matrices")
    if a.shape[1] != b.shape[0]:
        raise ValueError("the inner matrix dimensions must match")
    if a.device != b.device or a.dtype != b.dtype:
        raise ValueError("a and b must have the same device and dtype")
    if not a.is_cuda or not a.is_contiguous() or not b.is_contiguous():
        raise ValueError("a and b must be contiguous GPU tensors")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("this kernel accepts FP16 or BF16 inputs")

    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = lambda meta: (
        triton.cdiv(M, meta["BLOCK_M"])
        * triton.cdiv(N, meta["BLOCK_N"]),
    )
    optimized_matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )
    return c

## 6.2: Dense tile configuration

The dense wrappers share tile sizes and input checks. The tile-coordinate helper maps a flattened tile index to its matrix coordinates.

In [ ]:
# ch06-source: dense-configuration
@dataclass(frozen=True)
class Config:
    bm: int = 64
    bn: int = 64
    bk: int = 32
    warps: int = 4
    stages: int = 3
    group_m: int = 8

    def kwargs(self):
        return dict(BM=self.bm, BN=self.bn, BK=self.bk,
                    num_warps=self.warps, num_stages=self.stages)



def check_pair(a, b, allow_empty=False, allow_strided=False):
    if a.ndim != 2 or b.ndim != 2 or a.shape[1] != b.shape[0]:
        raise ValueError("expected A[M,K] and B[K,N]")
    if not a.is_cuda or a.device != b.device or a.dtype != b.dtype:
        raise ValueError("inputs must share a CUDA device and dtype")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("expected FP16 or BF16")
    if not allow_strided and (not a.is_contiguous() or not b.is_contiguous()):
        raise ValueError("this wrapper accepts contiguous inputs")
    if allow_strided and any(s <= 0 for t in (a, b) for s in t.stride()):
        raise ValueError("grouped inputs require positive strides")
    m, k = a.shape
    n = b.shape[1]
    if not allow_empty and min(m, n, k) == 0:
        raise ValueError("matrix dimensions must be positive")
    return m, n, k



def check_config(cfg):
    if any(x < 16 or x & (x - 1) for x in (cfg.bm, cfg.bn, cfg.bk)):
        raise ValueError("tile dimensions must be powers of two >= 16")
    if cfg.warps not in (4, 8) or cfg.stages < 1 or cfg.group_m < 1:
        raise ValueError("invalid warp, stage, or group count")



@triton.jit
def tile_coordinates(tile, nm, nn, GROUP: tl.constexpr):
    band = GROUP * nn
    first_m = (tile // band) * GROUP
    height = tl.minimum(nm - first_m, GROUP)
    local = tile % band
    return first_m + local % height, local // height

## Pointer-based ordinary and persistent schedules

The same pointer kernel uses either one tile per program or a persistent loop over several tiles.

In [ ]:
# ch06-source: pointer-kernel
@triton.jit
def pointer_kernel(A, B, C, M, N, K,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr,
                   GROUP: tl.constexpr, PERSISTENT: tl.constexpr):
    count = tl.cdiv(M, BM) * tl.cdiv(N, BN)
    first = tl.program_id(0)
    if PERSISTENT:
        stop = count
    else:
        stop = first + 1
    for tile in range(first, stop, tl.num_programs(0)):
        pm, pn = tile_coordinates(tile, tl.cdiv(M, BM),
                                  tl.cdiv(N, BN), GROUP)
        rows = pm * BM + tl.arange(0, BM)
        cols = pn * BN + tl.arange(0, BN)
        rk = tl.arange(0, BK)
        acc = tl.zeros((BM, BN), tl.float32)
        for block in range(tl.cdiv(K, BK)):
            ks = block * BK + rk
            av = tl.load(A + rows[:, None] * K + ks[None, :],
                         (rows[:, None] < M) & (ks[None, :] < K), 0)
            bv = tl.load(B + ks[:, None] * N + cols[None, :],
                         (ks[:, None] < K) & (cols[None, :] < N), 0)
            acc = tl.dot(av, bv, acc)
        tl.store(C + rows[:, None] * N + cols[None, :], acc,
                 (rows[:, None] < M) & (cols[None, :] < N))

## Descriptor transfers and dense launch wrappers

The descriptor kernel requests matrix tiles by logical coordinates. The wrapper chooses the schedule, prepares the operands, and returns a callable with its output buffer.

In [ ]:
# ch06-source: descriptor-kernel
@triton.jit
def descriptor_kernel(A, BT, C, M, N, K,
                      BM: tl.constexpr, BN: tl.constexpr,
                      BK: tl.constexpr, GROUP: tl.constexpr,
                      PERSISTENT: tl.constexpr):
    ad = tl.make_tensor_descriptor(A, [M, K], [K, 1], [BM, BK])
    bd = tl.make_tensor_descriptor(BT, [N, K], [K, 1], [BN, BK])
    cd = tl.make_tensor_descriptor(C, [M, N], [N, 1], [BM, BN])
    first = tl.program_id(0)
    count = tl.cdiv(M, BM) * tl.cdiv(N, BN)
    if PERSISTENT:
        stop = count
    else:
        stop = first + 1
    # The basic scheduler deliberately omits cross-tile flattening.
    for tile in range(first, stop, tl.num_programs(0)):
        pm, pn = tile_coordinates(tile, tl.cdiv(M, BM),
                                  tl.cdiv(N, BN), GROUP)
        acc = tl.zeros((BM, BN), tl.float32)
        for block in range(tl.cdiv(K, BK)):
            av = ad.load([pm * BM, block * BK])
            btv = bd.load([pn * BN, block * BK])
            acc = tl.dot(av, btv.T, acc)
        cd.store([pm * BM, pn * BN], acc.to(C.dtype.element_ty))



def install_descriptor_allocator(device):
    def allocate(size, alignment, stream):
        # CUDA allocation and use follow the caller's current stream.
        raw = torch.empty(size + alignment, dtype=torch.uint8,
                          device=device)
        offset = (-raw.data_ptr()) % alignment
        return raw[offset:offset + size]
    triton.set_allocator(allocate)



def prepare_dense(a, b, variant="pointer", cfg=Config(), cap=None):
    """Allocate once; return an enqueue callable and its output.

    Descriptor preparation includes B transpose, outside the callable.
    Re-run preparation when input storage, layout, or values change.
    """
    m, n, k = check_pair(a, b)
    check_config(cfg)
    if variant not in ("pointer", "persistent", "descriptor",
                        "descriptor_persistent"):
        raise ValueError("unknown dense variant")
    persistent = "persistent" in variant
    count = triton.cdiv(m, cfg.bm) * triton.cdiv(n, cfg.bn)
    if cap is None:
        cap = torch.cuda.get_device_properties(a.device).multi_processor_count
    if cap < 1:
        raise ValueError("cap must be positive")
    grid = (min(cap, count) if persistent else count,)
    c = torch.empty((m, n), device=a.device, dtype=a.dtype)
    kw = dict(GROUP=cfg.group_m, PERSISTENT=persistent, **cfg.kwargs())
    if variant.startswith("descriptor"):
        if torch.version.hip or torch.cuda.get_device_capability(a.device)[0] < 9:
            raise RuntimeError("descriptor example requires NVIDIA SM90+")
        if not hasattr(tl, "make_tensor_descriptor"):
            raise RuntimeError("installed Triton lacks tensor descriptors")
        if (k * a.element_size()) % 16 or (n * a.element_size()) % 16:
            raise ValueError("K and N row widths must align to 16 bytes")
        if a.data_ptr() % 16:
            raise ValueError("A base address must align to 16 bytes")
        bt = b.T.contiguous()
        install_descriptor_allocator(a.device)
        def run():
            with torch.cuda.device(a.device):
                return descriptor_kernel[grid](a, bt, c, m, n, k, **kw)
    else:
        def run():
            with torch.cuda.device(a.device):
                return pointer_kernel[grid](a, b, c, m, n, k, **kw)
    return run, c

## Full autotuned descriptor-persistent kernel

This version includes the autotuning and flattened loop used in the printed persistent implementation. We measure it separately from the controlled four-way comparison.

In [ ]:
# ch06-source: printed-persistent
def descriptor_allocator(
    size: int,
    alignment: int,
    stream: Optional[int],
):
    raw = torch.empty(
        size + alignment, device="cuda", dtype=torch.uint8,
    )
    offset = (-raw.data_ptr()) % alignment
    return raw[offset:offset + size]



@triton.jit
def grouped_tile_coordinates(
    tile_id,
    num_pid_m,
    num_pid_n,
    GROUP_M: tl.constexpr,
):
    num_pid_in_group = GROUP_M * num_pid_n
    group_id = tile_id // num_pid_in_group
    first_pid_m = group_id * GROUP_M
    group_size_m = min(
        num_pid_m - first_pid_m, GROUP_M,
    )

    pid_in_group = tile_id % num_pid_in_group
    pid_m = first_pid_m + pid_in_group % group_size_m
    pid_n = (
        tile_id % num_pid_in_group
    ) // group_size_m
    return pid_m, pid_n



def persistent_configs():
    return [
        triton.Config(
            {"BLOCK_M": 64, "BLOCK_N": 64,
             "BLOCK_K": 32, "GROUP_M": 8},
            num_warps=4,
            num_stages=2,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128,
             "BLOCK_K": 64, "GROUP_M": 8},
            num_warps=4,
            num_stages=3,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 256,
             "BLOCK_K": 64, "GROUP_M": 8},
            num_warps=8,
            num_stages=3,
        ),
        triton.Config(
            {"BLOCK_M": 128, "BLOCK_N": 128,
             "BLOCK_K": 128, "GROUP_M": 8},
            num_warps=8,
            num_stages=2,
        ),
    ]



@triton.autotune(
    configs=persistent_configs(),
    key=["M", "N", "K"],
)
@triton.jit
def persistent_descriptor_matmul_kernel(
    a_ptr,
    b_t_ptr,
    c_ptr,
    M,
    N,
    K,
    NUM_SMS: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    start_pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_tiles = num_pid_m * num_pid_n
    k_tiles = tl.cdiv(K, BLOCK_K)

    a_desc = tl.make_tensor_descriptor(
        a_ptr,
        shape=[M, K],
        strides=[K, 1],
        block_shape=[BLOCK_M, BLOCK_K],
    )
    b_desc = tl.make_tensor_descriptor(
        b_t_ptr,
        shape=[N, K],
        strides=[K, 1],
        block_shape=[BLOCK_N, BLOCK_K],
    )
    c_desc = tl.make_tensor_descriptor(
        c_ptr,
        shape=[M, N],
        strides=[N, 1],
        block_shape=[BLOCK_M, BLOCK_N],
    )

    store_tile_id = start_pid - NUM_SMS
    for tile_id in tl.range(
        start_pid,
        num_tiles,
        NUM_SMS,
        flatten=True,
    ):
        pid_m, pid_n = grouped_tile_coordinates(
            tile_id,
            num_pid_m,
            num_pid_n,
            GROUP_M,
        )
        m_offset = pid_m * BLOCK_M
        n_offset = pid_n * BLOCK_N

        accumulator = tl.zeros(
            (BLOCK_M, BLOCK_N), dtype=tl.float32,
        )
        for k_tile in range(0, k_tiles):
            k_offset = k_tile * BLOCK_K
            a = a_desc.load([m_offset, k_offset])
            b_t = b_desc.load([n_offset, k_offset])
            accumulator = tl.dot(
                a, b_t.T, accumulator,
            )

        store_tile_id += NUM_SMS
        store_pid_m, store_pid_n = (
            grouped_tile_coordinates(
                store_tile_id,
                num_pid_m,
                num_pid_n,
                GROUP_M,
            )
        )
        output = accumulator.to(
            c_ptr.dtype.element_ty,
        )
        c_desc.store(
            [store_pid_m * BLOCK_M,
             store_pid_n * BLOCK_N],
            output,
        )



def persistent_descriptor_matmul(a, b_t):
    if a.ndim != 2 or b_t.ndim != 2:
        raise ValueError("a and b_t must be matrices")
    if a.device != b_t.device or a.dtype != b_t.dtype:
        raise ValueError("a and b_t must match")
    if not a.is_cuda:
        raise ValueError("a and b_t must be CUDA tensors")
    if not a.is_contiguous() or not b_t.is_contiguous():
        raise ValueError("a and b_t must be contiguous")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("expected FP16 or BF16 inputs")

    triton.set_allocator(descriptor_allocator)
    M, K = a.shape
    N, K_b = b_t.shape
    if K != K_b or min(M, N, K) <= 0:
        raise ValueError("incompatible matrix dimensions")

    major, _ = torch.cuda.get_device_capability(a.device)
    if torch.version.hip or major < 9:
        raise RuntimeError("TMA requires compute capability 9.0+")

    element_bytes = a.element_size()
    if (K * element_bytes) % 16:
        raise ValueError("the K row width must be 16-byte aligned")
    if (N * element_bytes) % 16:
        raise ValueError("the C row width must be 16-byte aligned")
    if a.data_ptr() % 16 or b_t.data_ptr() % 16:
        raise ValueError("input pointers must be 16-byte aligned")

    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    num_sms = torch.cuda.get_device_properties(
        a.device,
    ).multi_processor_count

    def grid(meta):
        num_tiles = (
            triton.cdiv(M, meta["BLOCK_M"])
            * triton.cdiv(N, meta["BLOCK_N"])
        )
        return (min(num_sms, num_tiles),)

    persistent_descriptor_matmul_kernel[grid](
        a,
        b_t,
        c,
        M,
        N,
        K,
        NUM_SMS=num_sms,
    )
    return c

## 6.3: Split-K multiplication

Separate programs compute partial sums over K. A reduction kernel combines those partial sums into the output.

In [ ]:
# ch06-source: split-k
@triton.jit
def split_k_kernel(A, B, P, M, N, K, SPLITS: tl.constexpr,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    tile = tl.program_id(0)
    split = tl.program_id(1)
    nn = tl.cdiv(N, BN)
    rows = (tile // nn) * BM + tl.arange(0, BM)
    cols = (tile % nn) * BN + tl.arange(0, BN)
    total = tl.cdiv(K, BK)
    begin = split * total // SPLITS
    end = (split + 1) * total // SPLITS
    rk = tl.arange(0, BK)
    acc = tl.zeros((BM, BN), tl.float32)
    for block in range(begin, end):
        ks = block * BK + rk
        av = tl.load(A + rows[:, None] * K + ks[None, :],
                     (rows[:, None] < M) & (ks[None, :] < K), 0)
        bv = tl.load(B + ks[:, None] * N + cols[None, :],
                     (ks[:, None] < K) & (cols[None, :] < N), 0)
        acc = tl.dot(av, bv, acc)
    address = P + split * M * N + rows[:, None] * N + cols[None, :]
    tl.store(address, acc, (rows[:, None] < M) & (cols[None, :] < N))



@triton.jit
def reduce_splits(P, C, ELEMENTS, SPLITS: tl.constexpr,
                  BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    value = tl.full((BLOCK,), 0, tl.float32)
    for split in range(SPLITS):
        value += tl.load(P + split * ELEMENTS + offsets,
                         offsets < ELEMENTS, 0)
    tl.store(C + offsets, value, offsets < ELEMENTS)



def prepare_split_k(a, b, splits=8, cfg=Config()):
    m, n, k = check_pair(a, b)
    check_config(cfg)
    if not isinstance(splits, int) or not 1 <= splits <= triton.cdiv(k, cfg.bk):
        raise ValueError("splits must be in [1, ceil(K/BK)]")
    partial = torch.empty((splits, m, n), device=a.device,
                          dtype=torch.float32)
    c = torch.empty((m, n), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(m, cfg.bm) * triton.cdiv(n, cfg.bn), splits)
    def run():
        with torch.cuda.device(a.device):
            split_k_kernel[grid](a, b, partial, m, n, k,
                                 SPLITS=splits, **cfg.kwargs())
            return reduce_splits[(triton.cdiv(m * n, 256),)](
                partial, c, m * n, SPLITS=splits, BLOCK=256)
    return run, c

## 6.4: Grouped GEMM

The grouped kernel assigns one shared tile space to a list of products and loads the metadata for the product that owns each tile.

In [ ]:
# ch06-source: grouped
@triton.jit
def grouped_kernel(AP, BP, CP, SHAPES, STRIDES, ENDS,
                   PROBLEMS: tl.constexpr, IS_BF16: tl.constexpr,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    tile = tl.program_id(0)
    # ENDS is int64, so the loop-carried start must begin as int64 too.
    start = tl.cast(0, tl.int64)
    for problem in range(PROBLEMS):
        end = tl.load(ENDS + problem)
        m = tl.load(SHAPES + problem * 3)
        n = tl.load(SHAPES + problem * 3 + 1)
        k = tl.load(SHAPES + problem * 3 + 2)
        sam = tl.load(STRIDES + problem * 6)
        sak = tl.load(STRIDES + problem * 6 + 1)
        sbk = tl.load(STRIDES + problem * 6 + 2)
        sbn = tl.load(STRIDES + problem * 6 + 3)
        scm = tl.load(STRIDES + problem * 6 + 4)
        scn = tl.load(STRIDES + problem * 6 + 5)
        if IS_BF16:
            a = tl.load(AP + problem).to(tl.pointer_type(tl.bfloat16))
            b = tl.load(BP + problem).to(tl.pointer_type(tl.bfloat16))
            c = tl.load(CP + problem).to(tl.pointer_type(tl.bfloat16))
        else:
            a = tl.load(AP + problem).to(tl.pointer_type(tl.float16))
            b = tl.load(BP + problem).to(tl.pointer_type(tl.float16))
            c = tl.load(CP + problem).to(tl.pointer_type(tl.float16))
        while tile < end:
            local = tile - start
            nn = tl.cdiv(n, BN)
            rows = local // nn * BM + tl.arange(0, BM)
            cols = local % nn * BN + tl.arange(0, BN)
            rk = tl.arange(0, BK)
            acc = tl.zeros((BM, BN), tl.float32)
            for block in range(tl.cdiv(k, BK)):
                ks = block * BK + rk
                av = tl.load(a + rows[:, None] * sam + ks[None, :] * sak,
                             (rows[:, None] < m) & (ks[None, :] < k), 0)
                bv = tl.load(b + ks[:, None] * sbk + cols[None, :] * sbn,
                             (ks[:, None] < k) & (cols[None, :] < n), 0)
                acc = tl.dot(av, bv, acc)
            tl.store(c + rows[:, None] * scm + cols[None, :] * scn, acc,
                     (rows[:, None] < m) & (cols[None, :] < n))
            tile += tl.num_programs(0)
        start = end



def prepare_grouped(pairs, cfg=Config(), cap=None):
    check_config(cfg)
    pairs = tuple(pairs)
    if not pairs:
        return lambda: None, []
    shapes = [check_pair(a, b, allow_empty=True, allow_strided=True)
              for a, b in pairs]
    device, dtype = pairs[0][0].device, pairs[0][0].dtype
    if any(a.device != device or a.dtype != dtype for a, b in pairs):
        raise ValueError("all grouped problems must share device and dtype")
    outputs = [torch.empty((m, n), device=device, dtype=dtype)
               for m, n, k in shapes]
    ends, total = [], 0
    for m, n, k in shapes:
        total += triton.cdiv(m, cfg.bm) * triton.cdiv(n, cfg.bn)
        ends.append(total)
    strides = [s for (a, b), c in zip(pairs, outputs)
               for t in (a, b, c) for s in t.stride()]
    metadata = [torch.tensor(values, device=device, dtype=torch.int64)
                for values in ([a.data_ptr() for a, b in pairs],
                               [b.data_ptr() for a, b in pairs],
                               [c.data_ptr() for c in outputs],
                               [v for shape in shapes for v in shape],
                               strides, ends)]
    if cap is None:
        cap = torch.cuda.get_device_properties(device).multi_processor_count
    if cap < 1:
        raise ValueError("cap must be positive")
    def run():
        # Retain owners of raw addresses until all launches have finished.
        _owners = pairs, outputs
        if total:
            with torch.cuda.device(device):
                return grouped_kernel[(min(cap, total),)](
                    *metadata, PROBLEMS=len(pairs),
                    IS_BF16=dtype == torch.bfloat16, **cfg.kwargs())
    return run, outputs

## 6.5: Block-scaled operands

Quantization produces values and scales. The explicit provider reconstructs BF16 operands; the optional scaled-dot provider uses the compiler's target-dependent lowering.

In [ ]:
# ch06-source: block-scaled
def quantize_mxfp8_rows(x):
    """Educational finite-input quantizer; not a claim of OCP conformance.

    Returns E4M3 bytes [R,K], E8M0 exponent bytes [R,K/32].
    Rounds scales upward to a power of two to avoid finite-value overflow.
    """
    if x.ndim != 2 or x.shape[1] % 32 or min(x.shape) == 0:
        raise ValueError("expected nonempty rows with K divisible by 32")
    if not torch.isfinite(x).all().item():
        raise ValueError("quantizer accepts finite inputs only")
    blocks = x.float().reshape(x.shape[0], -1, 32)
    maximum = blocks.abs().amax(-1)
    exponent = torch.ceil(torch.log2(maximum.clamp_min(2.0 ** -100) / 448))
    exponent = torch.where(maximum == 0, 0, exponent)
    if ((exponent < -126) | (exponent > 127)).any().item():
        raise ValueError("scale exponent outside this example's range")
    scale = torch.exp2(exponent)
    q = (blocks / scale[..., None]).clamp(-448, 448)
    q = q.to(torch.float8_e4m3fn).reshape_as(x).contiguous()
    return q.view(torch.uint8), (exponent + 127).to(torch.uint8).contiguous()



def reconstruct_mxfp8_rows(values, scales):
    rows, k = values.shape
    q = values.view(torch.float8_e4m3fn).float().reshape(rows, k // 32, 32)
    factor = torch.exp2(scales.float() - 127)
    return (q * factor[..., None]).reshape(rows, k)



@triton.jit
def scaled_kernel(AQ, BTQ, AS, BS, C, M, N, K,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    tile = tl.program_id(0)
    rows = tile // tl.cdiv(N, BN) * BM + tl.arange(0, BM)
    cols = tile % tl.cdiv(N, BN) * BN + tl.arange(0, BN)
    rk = tl.arange(0, BK)
    rs = tl.arange(0, BK // 32)
    acc = tl.zeros((BM, BN), tl.float32)
    for block in range(tl.cdiv(K, BK)):
        ks = block * BK + rk
        ss = block * (BK // 32) + rs
        av = tl.load(AQ + rows[:, None] * K + ks[None, :],
                     (rows[:, None] < M) & (ks[None, :] < K), 0)
        bt = tl.load(BTQ + cols[:, None] * K + ks[None, :],
                     (cols[:, None] < N) & (ks[None, :] < K), 0)
        sa = tl.load(AS + rows[:, None] * (K // 32) + ss[None, :],
                     (rows[:, None] < M) & (ss[None, :] < K // 32), 127)
        sb = tl.load(BS + cols[:, None] * (K // 32) + ss[None, :],
                     (cols[:, None] < N) & (ss[None, :] < K // 32), 127)
        acc = tl.dot_scaled(av, sa, "e4m3", bt.T, sb, "e4m3", acc)
    tl.store(C + rows[:, None] * N + cols[None, :], acc,
             (rows[:, None] < M) & (cols[None, :] < N))



def prepare_scaled(aq, btq, sa, sb, cfg=Config(bk=128)):
    check_config(cfg)
    if not hasattr(tl, "dot_scaled"):
        raise RuntimeError("installed Triton lacks tl.dot_scaled")
    if any(t.ndim != 2 for t in (aq, btq, sa, sb)):
        raise ValueError("values and scales must be matrices")
    m, k = aq.shape
    n, kb = btq.shape
    if min(m, n, k) == 0 or kb != k or k % 32 or cfg.bk % 32:
        raise ValueError("incompatible shapes or scale group size")
    if sa.shape != (m, k // 32) or sb.shape != (n, k // 32):
        raise ValueError("scales must have shapes [M,K/32] and [N,K/32]")
    if any(t.device != aq.device or not t.is_cuda or
           t.dtype != torch.uint8 or not t.is_contiguous()
           for t in (aq, btq, sa, sb)):
        raise ValueError("values and scales need contiguous CUDA uint8 storage")
    if any((s == 255).any().item() for s in (sa, sb)):
        raise ValueError("NaN scales are outside this finite-input example")
    c = torch.empty((m, n), device=aq.device, dtype=torch.bfloat16)
    def run():
        with torch.cuda.device(aq.device):
            return scaled_kernel[(triton.cdiv(m, cfg.bm) *
                                   triton.cdiv(n, cfg.bn),)](
                aq, btq, sa, sb, c, m, n, k, **cfg.kwargs())
    return run, c

### Scale-layout exercise

This reversible permutation illustrates layout indexing. It is not a native hardware scale layout.

In [ ]:
import torch

logical = torch.arange(32).reshape(4, 8)
packed = logical.reshape(2, 2, 2, 4).permute(0, 2, 1, 3).contiguous()
restored = packed.permute(0, 2, 1, 3).contiguous().reshape(4, 8)
assert packed[0, 1, 1, 1].item() == logical[1, 5].item() == 13
assert torch.equal(restored, logical)

## 6.6: Block-sparse multiplication

The packed representation stores the present blocks and their indices. The multiplication visits those blocks while the validation functions check the representation.

In [ ]:
# ch06-source: block-sparse
@dataclass
class BlockRows:
    values: torch.Tensor
    row_ptr: torch.Tensor
    col_idx: torch.Tensor
    m: int
    k: int
    bm: int
    bk: int



def pack_block_rows(a, keep, bm=32, bk=32):
    """Apply an explicit block mask and pack retained blocks, including edges."""
    if a.ndim != 2 or min(a.shape) == 0 or not a.is_cuda:
        raise ValueError("expected a nonempty CUDA matrix")
    if a.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("expected FP16 or BF16")
    if bm < 16 or bk < 16 or bm & (bm - 1) or bk & (bk - 1):
        raise ValueError("block sizes must be powers of two >= 16")
    m, k = a.shape
    nr, nk = triton.cdiv(m, bm), triton.cdiv(k, bk)
    keep = torch.as_tensor(keep, device="cpu")
    if keep.dtype != torch.bool or keep.shape != (nr, nk):
        raise ValueError("keep must be a Boolean block mask")
    blocks, indices, row_ptr = [], [], [0]
    for row in range(nr):
        for col in range(nk):
            if keep[row, col]:
                block = torch.zeros((bm, bk), device=a.device, dtype=a.dtype)
                src = a[row * bm:min((row + 1) * bm, m),
                        col * bk:min((col + 1) * bk, k)]
                block[:src.shape[0], :src.shape[1]] = src
                blocks.append(block)
                indices.append(col)
        row_ptr.append(len(indices))
    values = (torch.stack(blocks) if blocks else
              torch.empty((0, bm, bk), device=a.device, dtype=a.dtype))
    return BlockRows(values,
                     torch.tensor(row_ptr, device=a.device, dtype=torch.int32),
                     torch.tensor(indices, device=a.device, dtype=torch.int32),
                     m, k, bm, bk)



def validate_block_rows(a):
    if min(a.m, a.k, a.bm, a.bk) <= 0:
        raise ValueError("invalid matrix or block shape")
    if a.bm < 16 or a.bk < 16 or a.bm & (a.bm-1) or a.bk & (a.bk-1):
        raise ValueError("block sizes must be powers of two >= 16")
    if not a.values.is_cuda or a.values.dtype not in (torch.float16, torch.bfloat16):
        raise ValueError("values must be FP16/BF16 on CUDA")
    for t in (a.row_ptr, a.col_idx):
        if t.device != a.values.device or t.dtype != torch.int32 or t.ndim != 1:
            raise ValueError("indices must be CUDA int32 vectors on values device")
    if any(not t.is_contiguous() for t in (a.values, a.row_ptr, a.col_idx)):
        raise ValueError("packed values and metadata must be contiguous")
    rp, ci = a.row_ptr.cpu().tolist(), a.col_idx.cpu().tolist()
    if len(rp) != triton.cdiv(a.m, a.bm) + 1 or rp[0] != 0:
        raise ValueError("invalid row pointer shape or first offset")
    if rp[-1] != len(ci) or any(x > y for x, y in zip(rp, rp[1:])):
        raise ValueError("row pointers must be monotone and end at nnzb")
    if a.values.shape != (len(ci), a.bm, a.bk):
        raise ValueError("packed value count or shape does not match metadata")
    for lo, hi in zip(rp, rp[1:]):
        row = ci[lo:hi]
        if any(c < 0 or c >= triton.cdiv(a.k, a.bk) for c in row):
            raise ValueError("K-block index out of bounds")
        if len(set(row)) != len(row):
            raise ValueError("duplicate blocks are not accepted")



def unpack_block_rows(a):
    validate_block_rows(a)
    dense = torch.zeros((a.m, a.k), device=a.values.device, dtype=a.values.dtype)
    rp, ci = a.row_ptr.cpu().tolist(), a.col_idx.cpu().tolist()
    for row in range(len(rp) - 1):
        for index in range(rp[row], rp[row + 1]):
            col = ci[index]
            h, w = min(a.bm, a.m - row*a.bm), min(a.bk, a.k-col*a.bk)
            dense[row*a.bm:row*a.bm+h, col*a.bk:col*a.bk+w] = a.values[index, :h, :w]
    return dense



@triton.jit
def sparse_kernel(V, RP, CI, B, C, M, N, K,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    total = tl.cdiv(M, BM) * tl.cdiv(N, BN)
    for tile in range(tl.program_id(0), total, tl.num_programs(0)):
        row = tile // tl.cdiv(N, BN)
        cols = tile % tl.cdiv(N, BN) * BN + tl.arange(0, BN)
        rows = row * BM + tl.arange(0, BM)
        rr, rk = tl.arange(0, BM), tl.arange(0, BK)
        begin, end = tl.load(RP + row), tl.load(RP + row + 1)
        acc = tl.zeros((BM, BN), tl.float32)
        for index in range(begin, end):
            kb = tl.load(CI + index)
            ks = kb * BK + rk
            av = tl.load(V + index * BM * BK + rr[:, None] * BK + rk[None, :],
                         (rows[:, None] < M) & (ks[None, :] < K), 0)
            bv = tl.load(B + ks[:, None] * N + cols[None, :],
                         (ks[:, None] < K) & (cols[None, :] < N), 0)
            acc = tl.dot(av, bv, acc)
        tl.store(C + rows[:, None] * N + cols[None, :], acc,
                 (rows[:, None] < M) & (cols[None, :] < N))



def prepare_sparse(a, b, bn=64, persistent=False):
    validate_block_rows(a)
    if b.ndim != 2 or b.shape[0] != a.k or b.shape[1] <= 0:
        raise ValueError("B must have shape [K,N] with N positive")
    if b.device != a.values.device or b.dtype != a.values.dtype or not b.is_contiguous():
        raise ValueError("B must match values device/dtype and be contiguous")
    cfg = Config(a.bm, bn, a.bk)
    check_config(cfg)
    n = b.shape[1]
    c = torch.empty((a.m, n), device=b.device, dtype=b.dtype)
    total = triton.cdiv(a.m, a.bm) * triton.cdiv(n, bn)
    cap = torch.cuda.get_device_properties(b.device).multi_processor_count
    grid = (min(total, cap) if persistent else total,)
    def run():
        with torch.cuda.device(b.device):
            return sparse_kernel[grid](a.values, a.row_ptr, a.col_idx,
                                       b, c, a.m, n, a.k, **cfg.kwargs())
    return run, c

## 6.7: References and timing helpers

The checks compare each output with a PyTorch reference. CUDA-event timing and synchronized wall-clock timing measure the two preparation policies used in the comparisons.

In [ ]:
# ch06-source: measurement
def reference(a, b):
    previous = torch.get_float32_matmul_precision()
    try:
        torch.set_float32_matmul_precision("highest")
        return a.float() @ b.float()
    finally:
        torch.set_float32_matmul_precision(previous)



def error_stats(actual, expected):
    delta = actual.float() - expected.float()
    if not torch.isfinite(actual).all().item():
        raise AssertionError("nonfinite output")
    if delta.numel() == 0:
        return dict(max_abs=0.0, relative_l2=0.0)
    return dict(max_abs=delta.abs().max().item(),
                relative_l2=(delta.norm() /
                             expected.float().norm().clamp_min(1e-12)).item())



def check(actual, expected):
    tol = 0.01 if actual.dtype == torch.float16 else 0.03
    torch.testing.assert_close(actual, expected.to(actual.dtype),
                               rtol=tol, atol=tol)
    return error_stats(actual, expected)



def inputs(m, n, k, dtype=torch.float16):
    return (torch.randn((m, k), device="cuda", dtype=dtype),
            torch.randn((k, n), device="cuda", dtype=dtype))



def device_ms(run):
    run()
    torch.cuda.synchronize()
    return float(triton.testing.do_bench(run, warmup=100, rep=300,
                                         return_mode="median"))



def call_ms(run, repeats=7):
    """Synchronized wall time includes Python preparation and CUDA work."""
    run()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        run()
        torch.cuda.synchronize()
        samples.append(1000 * (time.perf_counter() - start))
    return dict(median_ms=statistics.median(samples), samples_ms=samples)



def execute_prepared(prepare):
    run, out = prepare()
    run()
    return out



def selected_config(kernel):
    config = kernel.best_config
    return dict(**config.kwargs, num_warps=config.num_warps,
                num_stages=config.num_stages)

### Record the run

The recorder saves the environment, results, and generated PTX. It also saves the
executed source cells and their hashes so a result identifies the notebook code
that produced it. Rerunning a source cell replaces its earlier version in this record.

In [ ]:
# ch06-source: recorder
def capture_notebook_sources():
    """Return the last executed version of each marked source cell."""
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this notebook in a Jupyter/IPython kernel")
    sources = {}
    prefix = "# ch06-source: "
    for source in shell.history_manager.input_hist_raw:
        first_line = source.splitlines()[0] if source else ""
        if first_line.startswith(prefix):
            name = first_line[len(prefix):].strip()
            if name in CH06_SOURCE_CELLS:
                sources[name] = source
    missing = [name for name in CH06_SOURCE_CELLS if name not in sources]
    if missing:
        raise RuntimeError(
            "Run all definition cells in order before starting experiments; "
            f"missing source cells: {missing}"
        )
    return {name: sources[name] for name in CH06_SOURCE_CELLS}


class Recorder:
    def __init__(self, directory, target):
        self.path = Path(directory)
        self.path.mkdir(parents=True, exist_ok=False)
        self.rows = []
        self.target = target
        self.plan = target["features"]
        env = dict(**target,
                   seed=6, created_utc=datetime.now(timezone.utc).isoformat(),
                   timing="do_bench median, warmup=100ms, rep=300ms",
                   call_timing="7 synchronized wall-clock samples",
                   accuracy="FP16 rtol=atol=.01; BF16 .03; FP32 reference",
                   source_format="executed_notebook_cells")
        self.write("environment.json", env)
        sources = capture_notebook_sources()
        self.write("notebook_cells.json", sources)
        self.write("sources.json", {
            name: hashlib.sha256(source.encode("utf-8")).hexdigest()
            for name, source in sources.items()})

    def write(self, name, data):
        (self.path / name).write_text(json.dumps(data, indent=2),
                                     encoding="utf-8")

    def add(self, **row):
        self.rows.append(row)
        self.write("results.json", self.rows)
        print(json.dumps(row))

    def save_ptx(self, compiled, stem):
        if compiled is not None and hasattr(compiled, "asm"):
            ptx = compiled.asm.get("ptx", "")
            (self.path / f"{stem}.ptx").write_text(ptx, encoding="utf-8")

## Check correctness before timing

The checks cover dense, split-K, grouped, scaled, and sparse products, including the boundary cases accepted by their wrappers.

In [ ]:
# ch06-source: correctness
def correctness(rec):
    shapes = [(256, 256, 256), (257, 259, 263), (31, 47, 29),
              (769, 193, 257), (193, 769, 257), (128, 256, 32768)]
    for dtype in (torch.float16, torch.bfloat16):
        for m, n, k in shapes:
            a, b = inputs(m, n, k, dtype)
            ref = reference(a, b)
            rec.add(suite="correctness", provider="chapter5", shape=[m,n,k],
                    dtype=str(dtype), **check(optimized_matmul(a, b), ref))
            for variant in ("pointer", "persistent"):
                run, c = prepare_dense(a, b, variant)
                run()
                rec.add(suite="correctness", provider=variant, shape=[m,n,k],
                        dtype=str(dtype), **check(c, ref))
            for splits in (1, 2, 3, 8, 16):
                if splits > triton.cdiv(k, 32):
                    continue
                run, c = prepare_split_k(a, b, splits)
                run()
                rec.add(suite="correctness", provider="split_k", splits=splits,
                        shape=[m,n,k], dtype=str(dtype), **check(c, ref))
        descriptor_shapes = ([(31, 48, 40), (257, 264, 264),
                              (1024, 1024, 1024)]
                             if rec.plan["descriptors"]["enabled"] else [])
        for m,n,k in descriptor_shapes:
            a,b = inputs(m,n,k,dtype)
            for variant in ("descriptor", "descriptor_persistent"):
                run,c = prepare_dense(a,b,variant)
                run()
                rec.add(suite="correctness", provider=variant, shape=[m,n,k],
                        dtype=str(dtype), **check(c,reference(a,b)))
            c = persistent_descriptor_matmul(a,b.T.contiguous())
            rec.add(suite="correctness",provider="printed_persistent_listing",
                    shape=[m,n,k],dtype=str(dtype),**check(c,reference(a,b)))
        pairs = [inputs(*shape,dtype) for shape in
                 [(0,64,32),(31,47,29),(65,129,96),(8,16,0),(257,64,65)]]
        strided_a = torch.randn((29, 31), device="cuda", dtype=dtype).T
        strided_b = torch.randn((47, 29), device="cuda", dtype=dtype).T
        pairs.append((strided_a, strided_b))
        padded_a = torch.randn((33, 70), device="cuda", dtype=dtype)[:, ::2]
        padded_b = torch.randn((35, 82), device="cuda", dtype=dtype)[:, ::2]
        pairs.append((padded_a, padded_b))
        run, outputs = prepare_grouped(pairs, cap=7)
        run()
        for i, ((a,b), c) in enumerate(zip(pairs,outputs)):
            rec.add(suite="correctness", provider="grouped", problem=i,
                    dtype=str(dtype), **check(c,reference(a,b)))
        empty_run, empty_outputs = prepare_grouped([])
        empty_run()
        assert empty_outputs == []
        a,b = inputs(65,79,97,dtype)
        masks = [torch.zeros((3,4),dtype=torch.bool),
                 torch.ones((3,4),dtype=torch.bool),
                 torch.tensor([[1,0,1,0],[0,0,0,0],[0,1,0,1]],
                              dtype=torch.bool)]
        for mask in masks:
            packed = pack_block_rows(a,mask)
            dense = unpack_block_rows(packed)
            # Independently form the masked reference; do not trust unpack alone.
            expanded = mask.repeat_interleave(32,0).repeat_interleave(32,1)
            expected_a = a * expanded[:65,:97].to(a.device)
            torch.testing.assert_close(dense,expected_a,rtol=0,atol=0)
            for persistent in (False, True):
                run,c = prepare_sparse(packed,b,persistent=persistent)
                run()
                rec.add(suite="correctness", provider="sparse",
                        persistent=persistent, nnzb=int(mask.sum()),
                        dtype=str(dtype), **check(c,reference(expected_a,b)))
        packed = pack_block_rows(a,torch.ones((3,4),dtype=torch.bool))
        bad = packed.col_idx.clone()
        bad[0] = 4
        try:
            validate_block_rows(replace(packed,col_idx=bad))
        except ValueError:
            rec.add(suite="correctness", provider="sparse_invalid_index",
                    result="rejected")
        else:
            raise AssertionError("out-of-range sparse index was accepted")
        duplicate = packed.col_idx.clone()
        duplicate[1] = duplicate[0]
        try:
            validate_block_rows(replace(packed,col_idx=duplicate))
        except ValueError:
            rec.add(suite="correctness", provider="sparse_duplicate",
                    result="rejected")
        else:
            raise AssertionError("duplicate sparse index was accepted")

## Compare dense schedules

The dense comparison separates the common-configuration measurements from each implementation's independent search.

In [ ]:
# ch06-source: dense-comparison
def dense_experiment(rec):
    configs = [Config(), Config(128,128,64,4,3), Config(128,256,64,8,3)]
    variants = ["pointer", "persistent"]
    if rec.plan["descriptors"]["enabled"]:
        variants.extend(["descriptor", "descriptor_persistent"])
    for m,n,k in [(128,256,32768),(1024,1024,1024),(4096,4096,4096)]:
        a,b = inputs(m,n,k)
        ref = reference(a,b)
        # Chapter 5 baseline has its original, separate autotune search.
        baseline_run = lambda: optimized_matmul(a,b)
        check(baseline_run(),ref)
        rec.add(suite="dense", provider="chapter5_autotuned", shape=[m,n,k],
                ms=device_ms(baseline_run),
                config=selected_config(optimized_matmul_kernel),
                allocation="wrapper includes output")
        if rec.plan["descriptors"]["enabled"]:
            bt = b.T.contiguous()
            printed_run = lambda: persistent_descriptor_matmul(a,bt)
            check(printed_run(),ref)
            rec.add(suite="dense",provider="printed_persistent_listing",
                    shape=[m,n,k],ms=device_ms(printed_run),
                    config=selected_config(persistent_descriptor_matmul_kernel),
                    allocation="wrapper includes output; B transpose retained")
        for variant in variants:
            candidates = []
            for cfg in configs:
                run,c = prepare_dense(a,b,variant,cfg)
                try:
                    compiled = run()
                except OutOfResources as exc:
                    rec.add(suite="dense", provider=variant, shape=[m,n,k],
                            config=asdict(cfg), status="skipped",
                            reason=str(exc))
                    continue
                errors = check(c,ref)
                elapsed = device_ms(run)
                candidates.append((elapsed,cfg))
                rec.add(suite="dense", provider=variant, shape=[m,n,k],
                        config=asdict(cfg), ms=elapsed, **errors)
                stem = f"{variant}_{m}_{n}_{k}_{cfg.bm}_{cfg.bn}_{cfg.bk}"
                rec.save_ptx(compiled, stem)
            if not candidates:
                raise RuntimeError(f"no configuration fits {variant} on this GPU")
            best_ms,best_cfg = min(candidates,key=lambda x:x[0])
            rec.add(suite="dense_selected", provider=variant, shape=[m,n,k],
                    config=asdict(best_cfg), ms=best_ms,
                    call=call_ms(lambda: execute_prepared(
                        lambda: prepare_dense(a,b,variant,best_cfg))))

## Compare split counts

The split-K comparison measures prepared execution and the full call for the selected split counts.

In [ ]:
# ch06-source: split-comparison
def split_experiment(rec):
    for m,n,k in [(128,256,32768),(64,128,8192),(256,256,4096)]:
        a,b = inputs(m,n,k)
        ref = reference(a,b)
        ordinary,c0 = prepare_dense(a,b)
        ordinary()
        check(c0,ref)
        rec.add(suite="split", provider="ordinary_fixed", shape=[m,n,k],
                ms=device_ms(ordinary))
        check(optimized_matmul(a,b),ref)
        rec.add(suite="split",provider="chapter5_autotuned",shape=[m,n,k],
                ms=device_ms(lambda: optimized_matmul(a,b)))
        for splits in (1,2,4,8,16,32,64):
            run,c = prepare_split_k(a,b,splits)
            run()
            rec.add(suite="split", provider="split_k", shape=[m,n,k],
                    splits=splits, workspace_bytes=4*splits*m*n,
                    ms=device_ms(run), **check(c,ref),
                    call=call_ms(lambda: execute_prepared(
                        lambda: prepare_split_k(a,b,splits))))

## Compare grouped products

The grouped comparison measures a shared launch against separate products, including metadata preparation in the full-call timing.

In [ ]:
# ch06-source: grouped-comparison
def grouped_experiment(rec):
    cases = {
        "tiny": [(32,128,256)] * 32,
        "uneven": [(16,128,256),(63,256,512),(257,128,1024)] * 8,
        "one_large": [(16,128,256)] * 15 + [(1024,1024,1024)],
    }
    for name,shapes in cases.items():
        pairs = [inputs(*shape) for shape in shapes]
        for provider in ("separate_fixed", "grouped"):
            if provider == "grouped":
                run,outputs = prepare_grouped(pairs)
            else:
                prepared = [prepare_dense(a,b) for a,b in pairs]
                outputs = [c for _,c in prepared]
                def run():
                    for f,c in prepared:
                        f()
            run()
            for (a,b),c in zip(pairs,outputs):
                check(c,reference(a,b))
            rec.add(suite="grouped", case=name, provider=provider,
                    shapes=shapes, ms=device_ms(run))
        def prepare_separate():
            return [execute_prepared(lambda: prepare_dense(a,b))
                    for a,b in pairs]
        rec.add(suite="grouped_call", case=name,
                separate=call_ms(prepare_separate),
                grouped=call_ms(lambda: execute_prepared(
                    lambda: prepare_grouped(pairs))))

## Compare block-scaled providers

The scaled comparison distinguishes execution on prepared operands from calls that include quantization and reconstruction.

In [ ]:
# ch06-source: scaled-comparison
def scaled_experiment(rec):
    for distribution in ("normal", "unequal_blocks", "outlier"):
        a,b = inputs(256,256,1024,torch.bfloat16)
        if distribution == "unequal_blocks":
            scales = torch.tensor([1/64,64],device="cuda").repeat(16)
            a = (a.float().reshape(256,32,32)*scales[None,:,None]).reshape_as(a).to(a.dtype)
        elif distribution == "outlier":
            a[:,::32] *= 64
        aq,sa = quantize_mxfp8_rows(a)
        btq,sb = quantize_mxfp8_rows(b.T.contiguous())
        ar = reconstruct_mxfp8_rows(aq,sa)
        brt = reconstruct_mxfp8_rows(btq,sb)
        reconstructed = reference(ar,brt.T)
        original = reference(a,b)
        # An explicit software path runs regardless of tl.dot_scaled support.
        emulate,c_em = prepare_dense(ar.to(torch.bfloat16),
                                     brt.T.to(torch.bfloat16).contiguous())
        emulate()
        check(c_em,reconstructed)
        rec.add(suite="scaled", distribution=distribution,
                provider="explicit_bf16_reconstruction",
                arithmetic=check(c_em,reconstructed),
                quantization=error_stats(reconstructed,original),
                total_error=error_stats(c_em,original),
                ms=device_ms(emulate),
                bf16_original_ms=device_ms(lambda: torch.mm(a,b)))
        def fresh_emulated():
            qa,ssa = quantize_mxfp8_rows(a)
            qb,ssb = quantize_mxfp8_rows(b.T.contiguous())
            ra = reconstruct_mxfp8_rows(qa,ssa).to(torch.bfloat16)
            rb = reconstruct_mxfp8_rows(qb,ssb).T.to(torch.bfloat16).contiguous()
            return execute_prepared(lambda: prepare_dense(ra,rb))
        rec.add(suite="scaled_call", distribution=distribution,
                provider="explicit_bf16_reconstruction",
                call=call_ms(fresh_emulated))
        if not rec.plan["dot_scaled"]["enabled"]:
            continue
        run,c = prepare_scaled(aq,btq,sa,sb)
        compiled = run()
        rec.save_ptx(compiled, f"dot_scaled_{distribution}")
        provider = ("dot_scaled_bf16_emulation"
                    if rec.target["capability"][0] == 9
                    else "dot_scaled_target_dependent")
        rec.add(suite="scaled", distribution=distribution,
                provider=provider, instructions_verified=False,
                arithmetic=check(c,reconstructed),
                quantization=error_stats(reconstructed,original),
                total_error=error_stats(c,original), ms=device_ms(run))
        def fresh_scaled():
            qa,ssa = quantize_mxfp8_rows(a)
            qb,ssb = quantize_mxfp8_rows(b.T.contiguous())
            return execute_prepared(lambda: prepare_scaled(qa,qb,ssa,ssb))
        rec.add(suite="scaled_call", distribution=distribution,
                provider=provider,
                call=call_ms(fresh_scaled))
    # Tiny hand-checkable values: each dot has 32 unit products.
    a = torch.ones((17,32),device="cuda",dtype=torch.bfloat16)
    b = torch.ones((32,19),device="cuda",dtype=torch.bfloat16)
    aq,sa = quantize_mxfp8_rows(a)
    bq,sb = quantize_mxfp8_rows(b.T.contiguous())
    ar = reconstruct_mxfp8_rows(aq,sa).to(torch.bfloat16)
    br = reconstruct_mxfp8_rows(bq,sb).T.to(torch.bfloat16).contiguous()
    run,c = prepare_dense(ar,br)
    run()
    torch.testing.assert_close(c,torch.full_like(c,32),rtol=0,atol=0)
    rec.add(suite="scaled",case="known_ones",
            provider="explicit_bf16_reconstruction",result="pass")
    if rec.plan["dot_scaled"]["enabled"]:
        run,c = prepare_scaled(aq,bq,sa,sb)
        run()
        torch.testing.assert_close(c,torch.full_like(c,32),rtol=0,atol=0)
        rec.add(suite="scaled",case="known_ones",
                provider="dot_scaled",result="pass")

## Compare sparse block patterns

The sparse comparison varies density and block pattern, measuring both prepared execution and preparation with multiplication.

In [ ]:
# ch06-source: sparse-comparison
def sparse_experiment(rec):
    m,n,k = 512,512,1024
    a,b = inputs(m,n,k)
    nr,nk = m//32,k//32
    for density in (0.0,0.125,0.25,0.5,0.75,1.0):
        count = round(nr*nk*density)
        for pattern in ("spread", "concentrated"):
            mask = torch.zeros(nr*nk,dtype=torch.bool)
            if pattern == "spread":
                # Equalize retained counts across rows; vary columns by row.
                for j in range(count):
                    row = j % nr
                    col = (j // nr + row * 7) % nk
                    mask[row*nk+col] = True
            else:
                mask[:count] = True
            mask = mask.reshape(nr,nk)
            packed = pack_block_rows(a,mask)
            dense = unpack_block_rows(packed)
            ref = reference(dense,b)
            dense_run,dc = prepare_dense(dense,b)
            dense_run()
            check(dc,ref)
            rec.add(suite="sparse", provider="dense_same_values",
                    density=density, pattern=pattern, ms=device_ms(dense_run))
            for persistent in (False,True):
                run,c = prepare_sparse(packed,b,persistent=persistent)
                run()
                rec.add(suite="sparse", provider="block_sparse",
                        density=density, pattern=pattern,
                        persistent=persistent, ms=device_ms(run),
                        **check(c,ref))
            def fresh_sparse():
                p = pack_block_rows(a,mask)
                return execute_prepared(lambda: prepare_sparse(p,b))
            rec.add(suite="sparse_call",density=density,pattern=pattern,
                    call=call_ms(fresh_sparse))

In [ ]:
# ch06-source: runner
def run_all(directory, *, descriptors="auto", dot_scaled="auto"):
    target = inspect_target(descriptors=descriptors, dot_scaled=dot_scaled)
    torch.manual_seed(6)
    rec = Recorder(directory, target)
    try:
        for feature, selection in rec.plan.items():
            if not selection["enabled"]:
                rec.add(suite="availability", provider=feature,
                        status="skipped", reason=selection["reason"])
        correctness(rec)
        dense_experiment(rec)
        split_experiment(rec)
        grouped_experiment(rec)
        scaled_experiment(rec)
        sparse_experiment(rec)
    except Exception as exc:
        rec.write("status.json",dict(status="failed",error=repr(exc)))
        raise
    skips = sum(row.get("status") == "skipped" for row in rec.rows)
    rec.write("status.json",dict(
        status="complete_with_skips" if skips else "complete",
        rows=len(rec.rows), skipped=skips, instructions_verified=False))
    return rec.path

## Run the chapter experiments

Run every definition cell before starting the comparisons. For the pointer-based
examples and explicit block-scale reconstruction only, set `DESCRIPTORS = "off"`
and `DOT_SCALED = "off"` in the settings cell and rerun from that cell.

Use an unused output directory. Compilation and tuning can take several minutes.
Unavailable optional examples and resource-limited configurations are recorded as
skips. Unexpected compilation errors or incorrect outputs stop the run and preserve
partial results. Keep the traceback and investigate before changing tolerances.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
major, minor = target["capability"]
results_dir = OUTPUT_ROOT / f"CH06_SM{major}{minor}_results_{stamp}"
run_all(results_dir, descriptors=DESCRIPTORS, dot_scaled=DOT_SCALED)

## Inspect and archive the results

The results contain your environment, correctness errors, timings, executed source
cells, and generated PTX for the exported kernels. `complete_with_skips` records
omissions; `complete` means all selected cases finished without recorded skips.
Inspect the generated instructions and collect profiler evidence before attributing
a timing difference to TMA, occupancy, or native microscaling.

After a failure in the experiment cell, you can still archive the partial results
if the runner created its output directory. Keep new measurements separate from the
earlier GB10 and H100 records.

In [ ]:
import json
import shutil

status = json.loads((results_dir / "status.json").read_text())
print(status)
print(json.loads((results_dir / "environment.json").read_text()))
archive = shutil.make_archive(str(results_dir), "zip", results_dir)
print(archive)

## Third-party notice

Some implementations adapt the Triton matrix-multiplication, grouped-GEMM,
persistent-matmul, and block-scaled-matmul tutorials. Retain the following upstream
notice when distributing this notebook.

```text
/*
* Copyright 2018-2020 Philippe Tillet
* Copyright 2020-2022 OpenAI
*
* Permission is hereby granted, free of charge, to any person obtaining
* a copy of this software and associated documentation files
* (the "Software"), to deal in the Software without restriction,
* including without limitation the rights to use, copy, modify, merge,
* publish, distribute, sublicense, and/or sell copies of the Software,
* and to permit persons to whom the Software is furnished to do so,
* subject to the following conditions:
*
* The above copyright notice and this permission notice shall be
* included in all copies or substantial portions of the Software.
*
* THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
* EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
* MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
* IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
* CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
* TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
* SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
*/

```